
---
tags: ssh, globus, expanse
---

# Getting Started
## People
- Cindy Wong
- Andre Zonca - Chair of summer institute
- Susan Rathbun - Program Manager
- Nicole Wolter - Computational and Data Science Research Specialist

## Location
9836 Hopkins Drive, La Jolla CA, 92093

Code in Park Mobile: 
- 47800 
- $8 instead of $32
- Tickets are $80

## Details and Resources
- Allocation: CIS261077
- Reservations: si26cpu OR si26gpu
- Account:sdp173
- Materials: https://github.com/sdsc/sdsc-summer-institute-2026/tree/main
- Slack: https://app.slack.com/client/T0BHCRCNU68/C0BH2SBMPH9
- Videos:  https://www.sdsc.edu/education/on-demand-           
  learning/index.html
- git repos
  - https://github.com/sdsc/sdsc-summer-institute-2026.git
  - https://github.com/sdsc-hpc-training-org/basic_skills

Profile Link: https://allocations.access-ci.org/login
- Use ACCESS CI login not Cal Poly's CI logon (they are different)
- User: **jkrone**
- Password: see lastpass
- identity name: jkrone@access-ci.org
- subject id: f82c18ec-dd44-44e0-af07-f091684fceb6


### Login Node Access
Login nodes are meant for file editing, simple data analysis, and other tasks that use minimal compute resources. Use batch nodes for computationally intensive work.

```shell
# create ssh key
ssh-keygen -t ed25519 -C "jmkrone@calpoly.edu"

# add or change passphrase
ssh-keygen -p -f ~/.ssh/id_ed25519

# start ssh agent
eval "$(ssh-agent -s)"

# load key into agent
ssh-add ~/.ssh/id_ed25519_expanse

# verify key was added to agent
ssh-add -l

# setup MFA
# Login using Globus and Access ID
https://passive.sdsc.edu
# Add SSH key and MFA app in the website
https://passive.sdsc.edu/client/

# add your git key to the ssh-agent
ssh-add ~/.ssh/id_ed25519_github

# pass identities associated with ssh-agent through to remote
# do not copy ssh-key onto shared remote system
ssh -A jkrone@login.expanse.sdsc.edu

# setup host file b/c correct key may not be chosen by default
vim ~/.ssh/config

### On Local Machine
Host expanse
    HostName login.expanse.sdsc.edu
    User jkrone
    IdentityFile ~/.ssh/id_ed25519_expanse
  
Host github.com
   HostName github.com
   User git
   IdentityFile ~/.ssh/id_ed25519_github

### On Remote Machine (note the deliberate lack of identity file)
Host github.com
    HostName github.com
    User git

```
### Expanse User Portal
The [Expanse User Portal](https://portal.expanse.sdsc.edu/pun/sys/dashboard) provides a quick and easy way for Expanse users to log in, transfer and edit files, and submit and monitor jobs. The Portal provides a gateway for launching interactive applications such as MATLAB, RStudio, and an integrated web-based environment for file management and job submission. All ACCESS users with a valid Expanse allocation have access via their ACCESS-based credentials.


## Basic Linux Skills for Expanse
Fundamental Linux commands for navigating the file system, managing files, and understanding permissions.
```shell
date
hostname
whoami
env
groups

# recursively copy and preserve data and time
cp -r -p

# list by date
ls -alt

head -n 1
tail -n 1

# get line number and ignore case
grep -ni

# change permissions
chmod 660 *

# change group
chgrp heart *.out
```

Other utilities to learn
- grep, sort, tar, gzip ,and pigz

## Basic Orientation on Expanse

```shell
expanse-client user
expanse-client project sdp173
expanse-client project sdp173 -v
```
### Notes
You should use a **reservation** to get high priority in the queue
- si26cpu for most jobs
- si26gpu for GPU jobs

## Interactive Computing on Expanse
Requesting and using interactive sessions on CPU and GPU nodes.

### Request Interactive CPU Node
This node could be used for running jobs, jupyter notebooks, etc.

The following example will request one regular compute node, 4 cores, in the debug partition for 30 minutes.
```shell
srun --partition=debug  --pty --account=<<project>> --nodes=1 --ntasks-per-node=4 --mem=8G -t 00:30:00 --wait=0 --export=ALL /bin/bash
```

### Request Interactive GPU Node


```shell
srun --partition=gpu-shared --reservation=gputraining --nodes=1 --ntasks-per-node=6 --gres=gpu:k80:1  -t 03:00:00 --pty --wait=0 /bin/bash

# Once in the allocation, Load the CUDA and PGI compiler modules

module purge
module load gnutools
module load cuda
module load pgi

# If you get a license error when loading gpi compiler
export LM_LICENSE_FILE=40200@elprado.sdsc.edu:$LM_LICENSE_FILE
```

## Day 1

### Parallel Computing
- Very important
- Use atomic ints for counters
- Use mutexes for critial sections
- Data partioning is important for paralelizing
- Shared memory is helpful (internal memory aka registers)
- Cache 1, 2 Side 46
- You MUST be in level 2 cache, otherwise you're underutilizing the system
#### Summary

### Expanse
- https://expanse.sdsc.edu
- For jobs that run on one rack
- Liquid cooled rack
- Storage in Petabytes
- Hardware does fail. Have a hello world batch job to test
#### HPC System Architecture
- Login Node - simple tasks
- Computes Note - allocated by scheduler
- Internal Network - high bandwidth
- Shared network Filesystems - I/O
- Interactive Computing vs Batch Processing

#### Jupyter on Expanse
Install Galyleo

```shell
# Set env var to application that is aready installed
export PATH="/cm/shared/apps/sdsc/galyleo:${PATH}"

# helps you see which modules are avaliable - some have more libraries installed
module avali

# works
galyleo launch --account sdp173 --reservation si26cpu --partition compute --cpus 2 --memory 4 --time-limit 00:30:00 --env-modules cpu/0.17.3b,gcc/10.2.0,py-jupyterlab/3.2.1

# does not work
galyleo launch --account sdp173 --reservation si26cpu --partition compute --cpus 2 --memory 4 --time-limit 00:30:00 --env-modules cpu/0.17.3b,gcc/10.2.0,anaconda3/2021.05/q4munrg


```

### 2.3 High Throughput Computing
- Complecs
- HPC = Supercomputers and computer clusters to solve advanced computation problems
- Speed - 
- Scale - larger problems
- Throughput - solve many simple problems quickly

Usually - 2 CPUs per nodes on Expanse so parallelism is crucial!

#### High-Throughput Computing (HTC)
- Embarassingly parallel problems (small independent subtasks)
- Could run on laptops, but need to do it millions of times
- Brut-force, Batch processing, Montecarlo simulations
- Parameter sweeps

Advantags of HTC vs. HPC
- Simpler programming models - shared memory, serial
- Leverage distributed and heterogeneoud resources easily
- Resilient against job failures

#### Many-Task Computing (MTC)
- Distinct subtasks of variable complexity, often coupled with I/O operations (File Modeling Workflows)

#### Batch Scheduling
- #SBATCH directives tell scheduler these are the resources I want

#### Job Arrays
- Spinup multiple versions of the same job in one script `#SBATCH: array=1`

#### Job Dependencies
- Link post or pre processing steps to job

#### Job Bundling
- Nothing in slurm that will do it natively
- Most HPC systems don't allow sharing nodes (Expanse does allow partial node jobs)
- **Resource scheduling at the node-level**
- Allows more effective utilization

#### Distributed HTC Resources
- Open Science Grid (OSG) (Use to be PATh)
    - OS Pool is Free
    - Under 12 hour jobs
    - compute resources (CERN), HTCondor = scheduler
    - All jobs are small
    - portal.osg-htc.org
- National Research Platform
    - Nautilus
    - Lots of GPUs

#### Tools

Dask (parallel pandas)
- parallel and distributed computing. Custom schedulers that execute task graphs

Pegasus WMS
- Describe workflow in python and execute (like CDK)

Snakemake
- Bioinformatics
- Interact with slurm

Nextflow
- Bioinformatics
- Interact with slurm

Gnu Parallel
- https://hpc.njit.edu/Software/utilities/parallel/

#### Take Away

A lot of python libs are parallelized already you just need to turn on those options

#### Hands On

```shell

```

## Questions
- Support Model
  - Marty - Computational Physics - User Services
  - 1 FTE User Services and 1 FTE System Admin per system (Expanse, NRP)

- Datasets of interest
  - Alpha Fold

- How many people support daily operations of Expanse
- What does support usually look like for an academic seeking help?
- Where does UCSD's support model for Expanse start and stop?
  - For people
  - For system maintenance
- What is the support volume like? (100-200 people on system a day)
- Do you have insight into how much code run on Expanse uses parallelism? Is the system being used efficiently?

- Do you have any monitoring in place that helps you understand how big of a problem NOT paralellizing is, on Expanse?

- How do you ensure

Maya - behind